In [ ]:
import sys
from pathlib import Path

# Ensure project-root imports (e.g., utils.*) work regardless notebook launch directory.
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / 'utils').is_dir():
        if str(candidate) not in sys.path:
            sys.path.append(str(candidate))
        break

import numpy as np
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
import yaml
from utils.config import load_env_paths
from utils.mypalette import day_palette, pairing_strategy_palette

ENV_PATHS = load_env_paths(str((candidate / 'configs' / 'env.yml').resolve()))

order = ['Hand-crafted', 'Pretrained\nResNet18', 'Standard\nSimCLR', 'uniDINO', 'CellPaint SSL', 'SI\nSimCLR']

## ARI Scores for ALS:TIC

In [ ]:
res_dir = os.path.join(ENV_PATHS['analysis_results_root_dir'], 'ARI_UMAP', 'best_umap_kmeans_ari.yaml')

with open(res_dir, 'r') as f:
    data = yaml.safe_load(f)

# Flatten the nested structure
rows = []
for model_name, experiments in data.items():
    for exp_id, metrics in experiments.items():
        # Create a single dictionary for the row
        row = {'Model': model_name, 'Batch': exp_id}
        row.update(metrics) # Merges ari, k_used, etc.
        rows.append(row)

# Create the DataFrame
df = pd.DataFrame(rows)

df['Model'] = df['Model'].str.replace('hand-crafted', 'Hand-crafted')
df['Model'] = df['Model'].str.replace('Augmented\nSimCLR', 'SI\nSimCLR')

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 6))

sns.boxplot(x='Model', y='ari', data=df, ax=ax,
            order=order,
            palette=pairing_strategy_palette,
            hue='Model', showfliers=False
            )

sns.stripplot(x='Model', y='ari', data=df, ax=ax, palette=day_palette, dodge=False,
              marker='o', alpha=1, hue='Batch', edgecolor='black', linewidth=0.5, size=8, order=order)
# remove right and top spines
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
plt.title('ALS:TIC ARI')
plt.savefig('results/comparison_features/ARI scores/ALS_TIC_ARI.pdf', dpi=300, bbox_inches='tight')
plt.show()

## ARI Scores for batch

In [ ]:
res_dir = os.path.join(ENV_PATHS['analysis_results_root_dir'], 'ARI_UMAP_batch', 'best_umap_kmeans_ari.yaml')

with open(res_dir, 'r') as f:
    data = yaml.safe_load(f)

# Flatten the nested structure
rows = []
for model_name, experiments in data.items():
    for exp_id, metrics in experiments.items():
        # Create a single dictionary for the row
        row = {'Model': model_name, 'Batch': exp_id}
        row.update(metrics) # Merges ari, k_used, etc.
        rows.append(row)

# Create the DataFrame
df = pd.DataFrame(rows)

df['Model'] = df['Model'].str.replace('hand-crafted', 'Hand-crafted')
df['Model'] = df['Model'].str.replace('Augmented\nSimCLR', 'SI\nSimCLR')

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 6))

sns.barplot(x='Model', y='ari', data=df, ax=ax,
            order=order,
            palette=pairing_strategy_palette,
            hue='Model',
            #showfliers=False
            )

#sns.stripplot(x='Model', y='ari', data=df, ax=ax, palette=day_palette, dodge=False,
#              marker='o', alpha=1, hue='Batch', edgecolor='black', linewidth=0.5, size=8, order=order)

# remove right and top spines
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
plt.title('Batch ARI')
plt.savefig('results/comparison_features/ARI scores/batch_ARI.pdf', dpi=300, bbox_inches='tight')
plt.show()

## Shannon entropy of reduced embedding spaces

In [ ]:
from sklearn.decomposition import PCA
from utils.load_all_features import *
import matplotlib.pyplot as plt
import seaborn as sns
from utils.mypalette import pairing_strategy_palette

features, annotations = load_all_features()

In [ ]:
PCA_entropies = dict()

for name, feat in features.items():

    pca_ = PCA(20, copy=False)
    pca = pca_.fit(feat)

    S_pca= pca.singular_values_
    eigen_val_pca = S_pca**2 / np.sum(S_pca**2)
    eigen_val_pca[eigen_val_pca == 0] = np.nextafter(0, 1)
    entropy_pca = -1*np.sum(eigen_val_pca*np.log(eigen_val_pca))
    PCA_entropies[name] = entropy_pca
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

entropy_df = pd.DataFrame(list(PCA_entropies.items()), columns=['Model', 'Entropy'])

sns.barplot(x='Model', y='Entropy', data=entropy_df, ax=ax,
            palette=pairing_strategy_palette,
            order=list(pairing_strategy_palette.keys()))

plt.savefig('results/comparison_features/PCA_entropy.pdf')